# DATA 266 HW2.5 — GPU Assignment I: Precision, Bandwidth, and the Cost of Attention

This notebook orchestrates Parts A-F against the `src/` modules in this repository.
It is intended to be run **top to bottom on the RTX 4090 GPU lab workstation**
after `git clone`, environment setup, and CUDA-enabled PyTorch install (see
`README.md`). It was authored and reviewed on a Mac with no NVIDIA GPU, so it
has **not been executed** and contains no cell outputs; every GPU-dependent
result is produced only when this notebook (or the equivalent `scripts/*.sh`)
is run on the actual workstation.

Do not fabricate outputs by hand-editing this notebook. Run the cells for
real, then transcribe the observed values into `reports/METRICS.md` and
`reports/RUN_LOG.txt`.


In [ ]:
# Personal parameters (reported for consistency with prior DATA 266 assignments;
# this assignment measures hardware, not a trained model, so these do not
# otherwise affect the benchmarks below).
SID4 = 9486
SEED = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0
print(f"SID4={SID4}, SEED={SEED}, SLICE={SLICE}, HP_ID={HP_ID}, CLS_A={CLS_A}, CLS_B={CLS_B}")


## Setup and GPU Verification

Add the repository root to `sys.path` so the `src` package imports correctly
regardless of the notebook's working directory, then confirm CUDA and the
RTX 4090 are actually visible before running anything else.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)
if cuda_available:
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print(
        "No CUDA device detected. The remaining cells in this notebook require "
        "the RTX 4090 GPU lab workstation and will raise a clear RuntimeError "
        "if run here."
    )


## Part A — Onboarding and Provenance

Captures the complete `nvidia-smi -q` output plus measured hardware facts
(GPU name, UUID, driver, CUDA/PyTorch versions, VRAM, power limit), and saves
a snapshot of the vendor-documented RTX 4090 specifications used later for
"percent of theoretical peak" calculations. Measured and vendor-documented
values are kept in separate structures — see `src/system_info.py`.


In [ ]:
from src.system_info import capture_all, VENDOR_SPECS_RTX_4090

capture_all()  # writes results/system_info/*


In [ ]:
# Vendor-documented specifications (NOT measured on this machine) used as the
# theoretical-peak denominator throughout Parts B and C. Verify against the
# live NVIDIA RTX 4090 spec page and the Ada whitepaper before trusting these
# for the final report -- see the "verification_note" field.
VENDOR_SPECS_RTX_4090


## Part B — Precision and Achieved Throughput

Dense matmul at N = 1024, 4096, 8192, 16384 for FP32, TF32, FP16, and BF16.
Every run is warmed up, synchronized before/after timing, repeated a recorded
number of times, and tagged with the real GPU UUID. Also probes whether an
additional lower precision (FP8) is exposed by the installed PyTorch build.


In [ ]:
from src.precision_benchmarks import run_all as run_precision_benchmarks

run_precision_benchmarks()


In [ ]:
from src.plotting import plot_precision_scaling

plot_precision_scaling()


**Plateau discussion (fill in after running the cells above):** for each
precision, state the matrix size at which achieved TFLOPS plateaus and, using
the actual `results/precision/precision_benchmark_results.csv` data, explain
why the smaller matrix sizes fall short of peak throughput (e.g., fixed
kernel-launch overhead dominating at small N, too little work to saturate all
streaming multiprocessors, or a memory-bound regime at small N per the Part C
roofline). Do not state a plateau size before the benchmark has actually run.


## Part C — Bandwidth-Bound vs. Compute-Bound

A memory-bound elementwise addition and a compute-bound square matmul,
compared against the RTX 4090 roofline (theoretical peak FP32 FLOPS /
theoretical peak memory bandwidth) using arithmetic intensity.


In [ ]:
from src.bandwidth_benchmarks import run_all as run_bandwidth_benchmarks

run_bandwidth_benchmarks()


## Part D — Cost of Attention

Naive attention (materializes the full sequence-by-sequence attention matrix)
vs. PyTorch's fused `scaled_dot_product_attention`, at batch size 1, 1 head,
head dimension 64, bfloat16, under `torch.inference_mode()`. Sequence lengths
512 through 16384 are tested; CUDA out-of-memory errors are caught per length
so one failure does not stop the sweep, and the OOM boundary is refined with a
few extra probes rather than assumed.


In [ ]:
from src.attention_benchmarks import run_all as run_attention_benchmarks

run_attention_benchmarks()


In [ ]:
from src.plotting import plot_attention_memory

plot_attention_memory()


In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("results/attention/attention_summary.json").read_text())
print(json.dumps(summary, indent=2))


**What the fused kernel avoids materializing:** the fused
scaled-dot-product-attention backend (Flash/memory-efficient attention) never
materializes the full `[seq_len, seq_len]` attention probability matrix in GPU
memory. It processes queries, keys, and values in tiles, computing partial
attention scores and an online (running) softmax so only small per-tile score
buffers and running output/normalization statistics stay resident. Because
the `O(seq_len^2)` score and probability matrices -- and the extra GPU-memory
read/write traffic they require -- are avoided entirely, the fused kernel's
peak memory scales close to `O(seq_len)` instead of `O(seq_len^2)`, which is
why it keeps succeeding at sequence lengths where the naive implementation
runs out of memory.


## Part E — Sustained Load and Thermal Behaviour

Runs a ~20-minute sustained matmul load while sampling GPU clocks,
temperature, power draw, and utilization every 5 seconds. **This is a
long-running cell.** Prefer running `scripts/run_thermal_test.sh` from a
terminal on the workstation instead, so the GPU-lab reservation clock and the
notebook session are independent; the cell below is provided for convenience
if you do want to drive it from the notebook.


In [ ]:
from src.thermal_benchmark import run_sustained_load, analyze_thermal_log

# ~20 minutes by default. Reduce duration_s only for a smoke test; the real
# submission run must use the full 1200 seconds (20 minutes) required by the
# assignment.
thermal_csv = run_sustained_load(duration_s=1200, interval_s=5)


In [ ]:
thermal_analysis = analyze_thermal_log(thermal_csv)
thermal_analysis


In [ ]:
from src.plotting import plot_thermal

plot_thermal()


## Part F — Update the Reports

With every cell above executed for real on the RTX 4090 workstation:

1. Copy the observed values from `results/*/*.csv` and `results/*/*.json`
   into `reports/METRICS.md`, replacing every "To be measured" placeholder,
   including Table HW2.5.1.
2. Append one entry per run to `reports/RUN_LOG.txt` using the template
   already in that file, so every value in `METRICS.md` is traceable to a
   UUID-labelled run.
3. Fill in `provenance/reservation_record.md` and `provenance/gpu_hours.md`
   from the actual GPU lab reservation and session times.
4. Commit, push, and tag the completed assignment as `hw2-5` per the
   instructions in `README.md` -- only after the real results are in place.
